# 08 · Era and temporal sensitivity of the SMM model — Severe maternal morbidity MIMIC-IV (M2)

| Field | Content |
|---|---|
| Notebook sequence | 01: S1 · S2 · S3 · S4 · G1 → 02: S5 · G2 → 03: S6 · S7 · G3 · R1 · S8 → 04: S8 → 05: S9 (S7) → 06: S9 (S3) → 07: S9 · C1-C3 · R2 · R4 → **08: S9 · C4** → 09: S9 · C4 → 10: G4 · S10 · S11 |
| What it does | Checks whether the model learns *when* an admission happened (calendar era, ICD-9 or ICD-10 coding) instead of the severity of the patient, and how it behaves on patients of a later period. First exploration of criterion C4; the C4 result used in the classification comes from notebook 09 |
| Stages and gates | S7 · S9 · C4 (exploratory) |
| Inputs | Local PostgreSQL MIMIC-IV database (tables of `sql/`): `predictors_24h_v2`, `patients` (anchor-year group), `cohort_obstetric_v2` (ICD version) |
| Outputs | `data/outputs_era_temporal.txt` |
| Data use | Only aggregate results are shown and saved; nothing at patient level. Do not add cells that show or export rows |
| Time | About three times the main run of notebook 03, plus five short temporal fits |
| Next | `09_temporal_validation_smm.ipynb` |
| Spanish version | `08_era_temporal_validation_smm_es.ipynb` (same code) |

**What the results can and cannot show.** `anchor_year_group` is a three-year group assigned by MIMIC-IV to each **patient**, not an exact admission date, so it is an approximate calendar marker. The ICD-9 to ICD-10 change (October 2015) falls inside the 2014–2016 group, so era and coding version are partly confounded. This is one institution with few events in the early groups: the temporal validation is exploratory and **not** an external validation.

> **Run order.** Run all cells top to bottom. First `QUICK = True` for a smoke test (minutes; never report QUICK numbers), then `QUICK = False`.

## S1 · Governance and configuration — libraries, connection and settings

Fixes the seed, the outer and inner folds, the repeats, the bootstrap replicates and the reference counts `EXPECTED` frozen at gate G1. `QUICK = True` is a smoke test (minutes); its numbers are never reported.

In [ ]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False        # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- settings specific to notebook 08
ERA_ORDER    = ["2008 - 2010", "2011 - 2013", "2014 - 2016", "2017 - 2019", "2020 - 2022"]   # anchor_year_group, MIMIC-IV
MIN_TRAIN_EVENTS = 40       # a temporal fit is skipped (and reported as skipped) below this number of training events
MIN_TEST_EVENTS  = 15       # ... or below this number of test events
MIN_SLOPE_EVENTS = 20       # calibration slope is shown only for groups with at least this many events
EVAL_MODELS  = ["LR", "XGB", "Stack"]

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "en"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


## S7 · Predictors and S4 / G1 — predictor table, dictionary and calendar information

`predictors_24h_v2` has one row per obstetric hospitalization; `anchor_year_group` comes from `mimiciv.patients` and the ICD coding version from `mimiciv.cohort_obstetric_v2` (`has_icd9_dx`, `has_icd10_dx`). Only counts are shown; the next cell shows the gate-G1 decision.

In [ ]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()
print("Dictionary roles ->", {"main": VAR_MAIN, "missingness indicator in main model": IND_MAIN})

# Plausibility limits: out-of-range values become missing (prespecified limits of the predictor dictionary)
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

for v in set(IND_MAIN):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["y"] = df["smm_any"].astype(int)

# ---- calendar information (aggregated groups only; it never leaves this notebook)
era = pd.read_sql("SELECT subject_id, anchor_year_group FROM mimiciv.patients", engine).drop_duplicates("subject_id")
era = era.rename(columns={"anchor_year_group": "era"})
df = df.merge(era, on="subject_id", how="left")

ver = pd.read_sql("SELECT hadm_id, has_icd9_dx, has_icd10_dx FROM mimiciv.cohort_obstetric_v2", engine).drop_duplicates("hadm_id")
d9 = ver["has_icd9_dx"].fillna(False).astype(bool); d10 = ver["has_icd10_dx"].fillna(False).astype(bool)
ver["icd_ver"] = np.select([d9 & ~d10, d10 & ~d9, d9 & d10], ["ICD-9 only", "ICD-10 only", "mixed"], default="none")
df = df.merge(ver[["hadm_id", "icd_ver"]], on="hadm_id", how="left")
df["icd_ver"] = df["icd_ver"].fillna("none")

unknown = sorted(set(df["era"].dropna().unique()) - set(ERA_ORDER))
print("anchor_year_group values not in ERA_ORDER:", unknown if unknown else "none")

elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible deliveries with missing era:", int(elig["era"].isna().sum()))

In [ ]:
# Gate G1 decision: the loaded data must reproduce the reference counts frozen at the count audit (aggregates only)
g1 = pd.DataFrame([{"count": k, "loaded": v, "expected": EXPECTED[k],
                    "result": "reproduced" if v == EXPECTED[k] else "DIFFERS"} for k, v in chk.items()])
table(g1, title="Gate G1: reference counts")
table(pd.Series({"G1 decision": "passes: continue to the analyses" if (g1["result"] == "reproduced").all()
                 else "does NOT pass: review the tables of sql/ before fitting any model"}, name="decision"),
      title="Gate G1 decision")

## S6 · Prediction point and window — analysis datasets

| Name | Predictors |
|---|---|
| `main` | age, hemoglobin, creatinine, creatinine missingness indicator, five categorical admission variables (replicates notebooks 03/05) |
| `main_noadmin` | age, hemoglobin, creatinine, creatinine missingness indicator (no administrative categorical variables) |
| `main_plus_era` | `main` + anchor-year group (diagnostic only; not a deployable predictor) |

The three use the same eligible delivery hospitalizations in the same order, which makes the comparison paired.

In [ ]:
def make_spec(name):
    """main = replicate of notebooks 03/05; main_noadmin = without the five administrative categorical variables;
    main_plus_era = main plus the anchor-year group (diagnostic only, not a deployable predictor)."""
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    cat = [c for c in CAT_COLS if c in d.columns]
    if name == "main":
        pass
    elif name == "main_noadmin":
        cat = []
    elif name == "main_plus_era":
        cat = cat + ["era"]
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=translate("y", LANG))

for nm in ["main", "main_noadmin", "main_plus_era"]:
    sp = make_spec(nm); d = sp["data"]
    print(f"{nm:14s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={sp['cat']}")

## S8 · Modeling, validation and metrics — preprocessing, learners, nested tuning, stacking and calibration

Everything that learns from data stays inside the training part of each split (gate G2: no patient on both sides):

* `Winsor` clips each numeric column to the 0.5th–99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped cross-validation** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [ ]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=translate("", LANG)):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

## S8 · Modeling, validation and metrics — metrics, cluster bootstrap and paired differences

Discrimination, calibration and clinical usefulness are computed on the pooled out-of-fold predictions of each repeat and averaged over repeats. The bootstrap resamples **patients** (all admissions of a patient move together), so the intervals reflect repeated admissions. Differences between models use the **same** bootstrap samples (paired).

In [ ]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

## S9 · C4 — era-specific functions

Performance by group, temporal fits and paired refits. All metrics use patient-count weights, so every interval is a patient-level cluster bootstrap.

In [ ]:
def perf(y, p, w):
    """Discrimination and calibration of one set of predictions with patient-count weights."""
    pos = np.sum(w * y); neg = np.sum(w * (1 - y))
    nan = dict(auroc=np.nan, auprc=np.nan, oe=np.nan, slope=np.nan, citl=np.nan)
    if pos < 1 or neg < 1:
        return nan
    z = logit(p)
    return dict(auroc=roc_auc_score(y, p, sample_weight=w), auprc=average_precision_score(y, p, sample_weight=w),
                oe=float(pos / np.sum(w * p)), citl=citl(y, z, w),
                slope=calib_slope(y, z, w) if pos >= MIN_SLOPE_EVENTS else np.nan)

METS = ("auroc", "auprc", "oe", "slope", "citl")

def _ci(arr, d=3):
    a = np.asarray(arr, float); a = a[np.isfinite(a)]
    if len(a) < 20:
        return ""
    lo, hi = np.percentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def _fmt(v, d=3):
    return "n/a" if v is None or not np.isfinite(v) else f"{v:.{d}f}"

def cluster_weights(g, B, seed):
    """Yield B patient-level bootstrap weight vectors (all admissions of a patient move together)."""
    ug, inv = np.unique(g, return_inverse=True)
    rng = np.random.default_rng(seed)
    for _ in range(B):
        yield np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)

# ---- 1. performance by era and ICD version, from the out-of-fold predictions of the main analysis
def group_performance(res, groups, B, seed):
    y = res["y"].astype(float); g = res["g"]
    def one(w):
        out = {}
        for gname, mask in groups.items():
            for m in EVAL_MODELS:
                acc = [perf(y[mask], res["preds"][m][r][mask], w[mask]) for r in range(N_REPEATS)]
                for k in METS:
                    vals = [a[k] for a in acc]
                    out[(gname, m, k)] = np.nan if all(np.isnan(vals)) else float(np.nanmean(vals))
        return out
    point = one(np.ones(len(y)))
    boot = {}
    t0 = time.time()
    for b, w in enumerate(cluster_weights(g, B, seed)):
        for k, v in one(w).items():
            boot.setdefault(k, []).append(v)
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}

def report_groups(res, point, boot, groups):
    y = res["y"]
    rows = []
    for gname, mask in groups.items():
        n = int(mask.sum()); ev = int(y[mask].sum()); pat = len(set(res["g"][mask]))
        for m in EVAL_MODELS:
            rows.append({"group": gname, "model": m, "adm": n, "patients": pat, "events": ev, "prev": f"{ev/max(n,1):.4f}",
                         "AUROC": _fmt(point[(gname, m, "auroc")]) + _ci(boot[(gname, m, "auroc")]),
                         "AUPRC": _fmt(point[(gname, m, "auprc")]) + _ci(boot[(gname, m, "auprc")]),
                         "O/E": _fmt(point[(gname, m, "oe")], 2) + _ci(boot[(gname, m, "oe")], 2),
                         "slope": _fmt(point[(gname, m, "slope")], 2) + _ci(boot[(gname, m, "slope")], 2)})
    table(pd.DataFrame(rows))

In [ ]:
# ---- 2/3. temporal and ICD-transition validation: train on one block of calendar time, test on a later one
def temporal_plan(d):
    """Rolling origin by anchor_year_group (train on all earlier groups, test on one later group) and ICD-9 -> ICD-10.
    Patients who appear in the test block are removed from the training block (no patient on both sides)."""
    eidx = d["era"].map({e: i for i, e in enumerate(ERA_ORDER)})
    plan = []
    for k in range(1, len(ERA_ORDER)):
        te = np.where(eidx == k)[0]
        test_pat = set(d["subject_id"].values[te])
        tr = np.where((eidx < k).values & ~d["subject_id"].isin(test_pat).values)[0]
        plan.append((f"rolling: train {ERA_ORDER[0][:4]}-{ERA_ORDER[k-1][-4:]} -> test {ERA_ORDER[k]}", tr, te, f"era {ERA_ORDER[k]}"))
    te = np.where(d["icd_ver"].values == "ICD-10 only")[0]
    test_pat = set(d["subject_id"].values[te])
    tr = np.where((d["icd_ver"].values == "ICD-9 only") & ~d["subject_id"].isin(test_pat).values)[0]
    plan.append(("ICD transition: train ICD-9-only -> test ICD-10-only", tr, te, "ICD coding: ICD-10 only"))
    return plan

def temporal_fit(res, tr, te, seed):
    X, y, g, prep = res["X"], res["y"], res["g"], res["prep"]
    out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, seed)
    return {m: out[m] for m in EVAL_MODELS}, chosen

def temporal_eval(y_te, g_te, preds, B, seed):
    point = {}
    for m in EVAL_MODELS:
        for k, v in perf(y_te, preds[m], np.ones(len(y_te))).items():
            point[(m, k)] = v
    for k in ("auroc", "auprc"):
        point[("Stack-LR", k)] = point[("Stack", k)] - point[("LR", k)]
    boot = {}
    for w in cluster_weights(g_te, B, seed):
        cur = {}
        for m in EVAL_MODELS:
            for k, v in perf(y_te, preds[m], w).items():
                cur[(m, k)] = v
        for k in ("auroc", "auprc"):
            cur[("Stack-LR", k)] = cur[("Stack", k)] - cur[("LR", k)]
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
    return point, {k: np.array(v) for k, v in boot.items()}

def run_temporal(res, B, seed):
    d = res["spec"]["data"]; y = res["y"].astype(float); g = res["g"]
    results = []
    for i, (label, tr, te, grp) in enumerate(temporal_plan(d)):
        info = dict(label=label, group=grp, train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                    test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()))
        if info["train_events"] < MIN_TRAIN_EVENTS or info["test_events"] < MIN_TEST_EVENTS:
            info["skipped"] = True
            print(f"[{label}] SKIPPED (train events {info['train_events']}, test events {info['test_events']})")
            results.append(info); continue
        assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
        t0 = time.time()
        preds, chosen = temporal_fit(res, tr, te, SEED + 500 + i)
        print(f"[{label}] fitted in {(time.time()-t0)/60:.1f} min (train adm {len(tr)}, test adm {len(te)})")
        point, boot = temporal_eval(y[te], g[te], preds, B, SEED + 900 + i)
        info.update(skipped=False, point=point, boot=boot, chosen=chosen)
        results.append(info)
    return results

def report_temporal(results, cv_point, cv_boot):
    rows = []
    for info in results:
        base = {"design": info["label"], "train adm/pat/events": f"{info['train_adm']}/{info['train_pat']}/{info['train_events']}",
                "test adm/pat/events": f"{info['test_adm']}/{info['test_pat']}/{info['test_events']}"}
        if info["skipped"]:
            rows.append({**base, "model": "skipped (too few events)"}); continue
        for m in EVAL_MODELS:
            p, b = info["point"], info["boot"]
            rows.append({**base, "model": m,
                         "AUROC": _fmt(p[(m, "auroc")]) + _ci(b[(m, "auroc")]),
                         "AUPRC": _fmt(p[(m, "auprc")]) + _ci(b[(m, "auprc")]),
                         "O/E": _fmt(p[(m, "oe")], 2) + _ci(b[(m, "oe")], 2),
                         "slope": _fmt(p[(m, "slope")], 2) + _ci(b[(m, "slope")], 2),
                         "CITL": _fmt(p[(m, "citl")], 2) + _ci(b[(m, "citl")], 2),
                         "CV AUROC, same block": _fmt(cv_point.get((info["group"], m, "auroc"), np.nan))})
    table(pd.DataFrame(rows))
    print("\n    Paired difference Stack - LR in the later block (95% patient-level cluster-bootstrap interval)")
    rows = []
    for info in results:
        if info["skipped"]:
            continue
        p, b = info["point"], info["boot"]
        rows.append({"design": info["label"],
                     "AUROC diff": f"{p[('Stack-LR','auroc')]:+.3f}" + _ci(b[("Stack-LR", "auroc")]),
                     "AUPRC diff": f"{p[('Stack-LR','auprc')]:+.3f}" + _ci(b[("Stack-LR", "auprc")])})
    table(pd.DataFrame(rows))

In [ ]:
# ---- 4. refit without / with era-related variables (paired comparison, same splits, same patient resamples)
def paired_alt(res_ref, alts, B, seed):
    """alts = {tag: res}. Returns point estimates, bootstrap draws, and differences (alt - ref)."""
    y = res_ref["y"].astype(float); g = res_ref["g"]
    for tag, r in alts.items():
        assert np.array_equal(r["y"], res_ref["y"]) and np.array_equal(r["g"], res_ref["g"]), "analyses must share rows and patients"
    models = ["LR", "RF", "ET", "XGB", "Stack"]
    mets = ("auroc", "auprc", "slope", "oe")
    allres = {"ref": res_ref, **alts}
    def one(w):
        out = {}
        for tag, res in allres.items():
            for m in models:
                acc = [perf(y, res["preds"][m][r], w) for r in range(N_REPEATS)]
                for k in mets:
                    out[(m, tag, k)] = float(np.mean([a[k] for a in acc]))
        return out
    point = one(np.ones(len(y)))
    boot = {}
    t0 = time.time()
    for b, w in enumerate(cluster_weights(g, B, seed)):
        cur = one(w)
        for k, v in cur.items():
            boot.setdefault(k, []).append(v)
        for tag in alts:
            for m in models:
                for k in mets:
                    boot.setdefault((m, f"diff_{tag}", k), []).append(cur[(m, tag, k)] - cur[(m, "ref", k)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return point, {k: np.array(v) for k, v in boot.items()}, models

def report_paired(res_ref, alts, point, boot, models):
    y = res_ref["y"]; d = res_ref["spec"]["data"]
    print("main          : numeric", res_ref["spec"]["num"], "binary", res_ref["spec"]["bin"], "categorical", res_ref["spec"]["cat"])
    for tag, r in alts.items():
        print(f"{tag:14s}: numeric", r["spec"]["num"], "binary", r["spec"]["bin"], "categorical", r["spec"]["cat"])
    for met, dd in (("auroc", 3), ("auprc", 3)):
        rows = []
        for m in models:
            r_ = {"model": m, f"{met} main": f"{point[(m,'ref',met)]:.{dd}f}"}
            for tag in alts:
                r_[f"{tag}"] = f"{point[(m,tag,met)]:.{dd}f}"
                r_[f"diff ({tag} - main)"] = f"{point[(m,tag,met)] - point[(m,'ref',met)]:+.{dd}f}" + _ci(boot[(m, f"diff_{tag}", met)], dd)
            rows.append(r_)
        print(f"\n    {met.upper()} (paired, 95% patient-level cluster-bootstrap intervals for the differences)")
        table(pd.DataFrame(rows))
    rows = []
    for m in models:
        r_ = {"model": m, "slope main": f"{point[(m,'ref','slope')]:.2f}", "O/E main": f"{point[(m,'ref','oe')]:.2f}"}
        for tag in alts:
            r_[f"slope {tag}"] = f"{point[(m,tag,'slope')]:.2f}"; r_[f"O/E {tag}"] = f"{point[(m,tag,'oe')]:.2f}"
        rows.append(r_)
    print("\n    Calibration after recalibration (slope; observed/expected)")
    table(pd.DataFrame(rows))
    for tag, r in [("main", res_ref)] + list(alts.items()):
        ch = pd.DataFrame(r["chosen"], columns=["learner", "config"])
        tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
        print(f"\n    Hyperparameters selected in the inner loops, {tag} (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
        table(tab)

In [ ]:
def emit(name, fn):
    """Write the aggregate report data/outputs_<name>.txt (English text, as cited by the article) and show it as tables;
    the Spanish notebook also writes data/outputs_<name>_es.txt. Aggregates only."""
    path = f"../data/outputs_{name}.txt"
    text = _emit_to(path, fn)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

## S9 · Sensitivity and robustness — run the analyses

Each cell prints the projected time after the first fold. The C4 result used in the classification comes from notebook 09.

### 7.1 `main` (replicate of notebooks 03/05)

In [ ]:
res_main = run_cv(make_spec("main"))
sp = res_main["spec"]
print("main predictors:", sp["num"], sp["bin"], sp["cat"])

### 7.2 `main_noadmin` — S7: contribution of the administrative variables

In [ ]:
res_noadm = run_cv(make_spec("main_noadmin"))

### 7.3 `main_plus_era` — S9 diagnostic: era as a predictor

In [ ]:
res_era = run_cv(make_spec("main_plus_era"))

### 7.4 Performance by anchor-year group and ICD coding version — S9 · C4 (exploratory)

From the out-of-fold predictions of `main`.

In [ ]:
d_main = res_main["spec"]["data"]
groups = {"All eligible deliveries": np.ones(len(d_main), bool)}
for e in ERA_ORDER:
    groups[f"era {e}"] = (d_main["era"] == e).values
for v in ["ICD-9 only", "mixed", "ICD-10 only"]:
    groups[f"ICD coding: {v}"] = (d_main["icd_ver"] == v).values
groups = {k: m for k, m in groups.items() if int(m.sum()) > 0}
pt_grp, bt_grp = group_performance(res_main, groups, BOOT_SENS, SEED + 31)

### 7.5 Rolling-origin temporal validation and ICD-9 to ICD-10 transition — S9 · C4 (exploratory)

For each later group, every learner is tuned, stacked and recalibrated **only** on the earlier groups and scored on that group; patients with admissions in the test group are removed from training. A fit is skipped when the training block has fewer than `MIN_TRAIN_EVENTS` or the test block fewer than `MIN_TEST_EVENTS` events.

In [ ]:
temporal = run_temporal(res_main, BOOT_SENS, SEED + 41)

### 7.6 Paired comparison of `main` with the two refits

In [ ]:
alts = {"no_admin_vars": res_noadm, "plus_era": res_era}
pt_pair, bt_pair, models_pair = paired_alt(res_main, alts, BOOT, SEED + 51)

## S11 · Reporting — report

Shows the aggregate tables and saves them as `data/outputs_era_temporal.txt`.

| Result | What is done | In this model |
|---|---|---|
| Removing the administrative variables or adding the era barely changes the AUROC | The model is not learning the era; keep the main predictors | −0.016 to −0.018 without them; +0.011 to +0.016 with the era |
| Calibration shifts in later groups | Expected with a rising SMM rate; C4 is decided in notebook 09 | O/E 0.69 in 2008–2010 and 1.19 in 2014–2016 |

In [ ]:
def full_report():
    y = res_main["y"]; d = res_main["spec"]["data"]
    print("=" * 110)
    print(f"ERA / TEMPORAL SENSITIVITY: SMM, eligible deliveries, n={len(y)}, patients={d.subject_id.nunique()}, "
          f"events={int(y.sum())}, prevalence={y.mean():.4f}")
    print(f"outer {OUTER_FOLDS}x{N_REPEATS}, inner {INNER_FOLDS}, bootstrap B={BOOT_SENS} (groups, temporal) / {BOOT} (paired refits), QUICK={QUICK}")
    print("=" * 110)

    print("\n[0] Eligible deliveries by anchor-year group and by ICD coding version (aggregate counts)")
    rows = []
    for key, lab in (("era", "anchor_year_group"), ("icd_ver", "ICD coding")):
        t = d.groupby(key, dropna=False).agg(adm=("hadm_id", "size"), patients=("subject_id", "nunique"), events=("y", "sum"))
        t["prev"] = (t.events / t.adm).round(4)
        t.insert(0, "variable", lab); t.index.name = "group"
        rows.append(t.reset_index())
    table(pd.concat(rows))
    ct = pd.crosstab(d["era"].fillna("missing"), d["icd_ver"])
    print("\n    Admissions by anchor-year group (rows) and ICD coding version (columns)")
    table(ct, index=True)

    print("\n[1] Cross-validated performance by group (out-of-fold predictions of the main analysis; patients are in the same folds as notebooks 03/05)")
    print("    O/E and slope come from probabilities calibrated on the whole development sample, so O/E far from 1 in one group means a calibration shift between groups.")
    report_groups(res_main, pt_grp, bt_grp, groups)

    print("\n[2] Temporal validation (rolling origin) and ICD-9 -> ICD-10 transition: models fitted only on the earlier block, scored on the later one")
    print("    Patients with admissions in the later block are removed from the earlier block. Each model is tuned, stacked and recalibrated inside the earlier block only.")
    report_temporal(temporal, pt_grp, bt_grp)

    print("\n[3] Refits without the administrative variables that can reflect the era, and with the era added as a diagnostic")
    report_paired(res_main, alts, pt_pair, bt_pair, models_pair)

emit("era_temporal", full_report)

## Notebook summary

Contribution of the administrative variables and of the era (first exploration of criterion C4).

In [ ]:
# Summary of notebook 08
notebook_summary([
    ("AUROC XGBoost, main", lambda: f"{_cv_auroc(res_main, 'XGB'):.3f}"),
    ("AUROC XGBoost, without the administrative variables", lambda: f"{_cv_auroc(res_noadm, 'XGB'):.3f}"),
    ("AUROC XGBoost, with the era as a predictor", lambda: f"{_cv_auroc(res_era, 'XGB'):.3f}"),
    ("criterion C4 used in the classification", "notebook 09"),
])